# VEILINK 六轴机械臂
# 三、单舵机控制

本 Notebook 从同目录下的 `机械臂校准参数.json` 加载 ID、零位和软限位，为 J1～J6 和夹爪创建独立图形化控制面板。

界面采用安全的两步操作：

1. 拖动滑块只修改目标值，不会立即运动；
2. 只有对应关节已经使能，并点击“移动到目标”后，才发送位置命令。

每个关节可独立执行：读取状态、使能保持、移动、设置零位目标和卸力。页面底部提供“全部紧急卸力”。

## 0. 安全要求

1. 将机械臂固定在稳定底座上，首次测试时不要安装负载。
2. 先运行“读取状态”，确认显示位置与实际姿态基本一致，再使能关节。
3. J2、J3 是承重关节。卸力前必须托住机械臂，防止连杆下坠。
4. 一次只测试一个关节；确认方向、零位和限位正确后再测试下一个。
5. 手不要放在夹爪、关节缝隙或连杆扫过区域。
6. 出现异常运动、发热、异响、通信错误或结构干涉时，立即点击“全部紧急卸力”并切断舵机电源。
7. J1/J4/J6 虽可整圈旋转，但外部线缆可能缠绕。本界面只提供一个编码器单圈内的 `-179.9°～179.9°` 目标。
8. 本 Notebook 只在内存中实施软件限位；即使舵机 EEPROM 已有限位，也不能省略软件检查。

## 1. 加载环境和校准参数

选择 `Python (lerobot)` kernel 后运行下一格。若参数文件的 `calibrated` 不是 `true`，程序将拒绝继续。

In [ ]:
import json
import sys
from importlib.metadata import version
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

WORK_DIR = Path.cwd()
if WORK_DIR.name == "Code":
    NOTEBOOK_DIR = Path(".")
elif (WORK_DIR / "Code").is_dir():
    NOTEBOOK_DIR = Path("Code")
else:
    NOTEBOOK_DIR = Path(".")

PARAM_PATH = NOTEBOOK_DIR / "机械臂校准参数.json"
if not PARAM_PATH.exists():
    raise FileNotFoundError(f"找不到校准参数文件：{PARAM_PATH}")

params = json.loads(PARAM_PATH.read_text(encoding="utf-8"))
if params.get("schema_version") != 1:
    raise ValueError("不支持的校准参数版本。")
if not params.get("calibrated"):
    raise RuntimeError("校准参数 calibrated=false，禁止控制舵机。请先完成 2-舵机校准.ipynb。")
KINEMATICS_REVISION = "2026-08-09-dh-v2"
if params.get("kinematics_status") != "final_confirmed" or params.get("kinematics_revision") != KINEMATICS_REVISION:
    raise RuntimeError("运动学参数不是 2026-08-09-dh-v2，请先重新运行校准 Notebook 的参数同步与保存单元。")

expected_dh_signature = [
    ("J1", 0, -90), ("J2", 90, 180), ("J3", 90, -90),
    ("J4", 180, 90), ("J5", 0, -90), ("J6", 0, 0),
]
actual_dh_signature = [
    (row.get("joint"), row.get("theta_offset_deg"), row.get("alpha_deg"))
    for row in params.get("dh_parameters", [])
]
if actual_dh_signature != expected_dh_signature:
    raise RuntimeError(f"DH参数与 2026-08-09-dh-v2 不一致：{actual_dh_signature}")

unverified_arm_directions = [
    joint for joint in ["J1", "J2", "J3", "J4", "J5", "J6"]
    if int(params["joints"][joint].get("direction", 0)) != 1
    or not params["joints"][joint].get("direction_verified", False)
]
if unverified_arm_directions:
    raise RuntimeError(f"以下运动关节未满足红色箭头对应 direction=+1 的最终方向约定：{unverified_arm_directions}")
if params["hardware"].get("gripper_enabled", False) and not params["joints"]["gripper"].get("direction_verified", False):
    raise RuntimeError("夹爪舵机方向尚未确认，拒绝进入控制流程。")
else:
    print("J1～J6及夹爪的舵机正方向均已确认。首次实机动作仍只允许2°～5°或2%～5%的小行程。")

print("Python:", Path(sys.executable).name)
print("LeRobot:", version("lerobot"))
print("校准文件：", PARAM_PATH)
print("校准时间：", params.get("calibrated_at"))


## 2. 检查端口和控制配置

默认使用校准文件记录的端口。若控制板的 COM 号发生变化，只修改这里的 `PORT`，不要修改校准 JSON。

`DEFAULT_SPEED` 和 `DEFAULT_ACCELERATION` 是发送到 STS3215 的运行参数。首次测试采用较低值；确认运动正常后再逐步提高。

In [ ]:
from serial.tools import list_ports

detected_ports = list(list_ports.comports())
if detected_ports:
    print("检测到的串口：")
    for port in detected_ports:
        print(f"  {port.device:8s} | {port.description}")
else:
    print("未检测到串口，请检查控制板、USB 线和驱动。")

# TODO：如果串口发生变化，在这里覆盖。
PORT = params["hardware"].get("port_used") or "COM3"

# 首次测试建议保持较低速度和加速度。STS3215 原始速度/加速度参数。
DEFAULT_SPEED = 200
DEFAULT_ACCELERATION = 20

JOINT_ORDER = ["J1", "J2", "J3", "J4", "J5", "J6"]
if params["hardware"].get("gripper_enabled", False):
    JOINT_ORDER.append("gripper")

print("PORT =", PORT)
print("舵机：", JOINT_ORDER)
print("默认速度/加速度：", DEFAULT_SPEED, DEFAULT_ACCELERATION)


## 3. 位置换算与软件限位

J1～J6 的界面单位是度，夹爪单位是百分比。所有关节目标必须同时满足最终运动学图片给出的设计范围和实测软限位，即使用两者交集。J6 按最终表显示为 `[0°,360°)`；J1/J4 使用 `[-180°,180°)`。界面角度是物理关节角 $q_i$，DH 固定偏置（如 $\theta_2=q_2+90^\circ$）只在 FK/IK 内使用，不应重复加到舵机目标上。

In [ ]:
ENCODER_RESOLUTION = int(params["hardware"].get("encoder_resolution", 4096))
ENCODER_MAX = ENCODER_RESOLUTION - 1

def wrapped_tick_delta(value, reference):
    return ((int(value) - int(reference) + ENCODER_RESOLUTION // 2) % ENCODER_RESOLUTION) - ENCODER_RESOLUTION // 2

def design_bounds(joint):
    item = params["joints"][joint]
    return float(item["design_min_deg"]), float(item["design_max_deg"])

def raw_to_display_value(joint, raw_value):
    item = params["joints"][joint]
    raw_value = int(raw_value)
    if joint == "gripper":
        low, high = int(item["soft_min_raw"]), int(item["soft_max_raw"])
        ratio = (raw_value - low) / (high - low)
        percent = ratio * 100.0 if int(item.get("direction", 1)) == 1 else (1.0 - ratio) * 100.0
        return max(0.0, min(100.0, percent))

    delta = wrapped_tick_delta(raw_value, item["zero_position_raw"])
    angle = int(item.get("direction", 1)) * delta * 360.0 / ENCODER_RESOLUTION
    # 最终参数表将 J6 发布为 [0,360)，将编码器的负半圈显示转换为等价正角度。
    if joint == "J6" and angle < 0.0:
        angle += 360.0
    return angle

def display_value_to_raw(joint, value):
    item = params["joints"][joint]
    if joint == "gripper":
        value = max(0.0, min(100.0, float(value)))
        low, high = int(item["soft_min_raw"]), int(item["soft_max_raw"])
        ratio = value / 100.0
        if int(item.get("direction", 1)) == -1:
            ratio = 1.0 - ratio
        raw = round(low + ratio * (high - low))
    else:
        value = float(value)
        design_low, design_high = design_bounds(joint)
        # 上端 360°/180° 与下端是同一单圈位置，界面最大值会减去0.1°。
        if value < design_low or value > design_high:
            raise ValueError(f"{joint} 目标 {value}° 超出最终设计范围 [{design_low}, {design_high}]")
        direction = int(item.get("direction", 1))
        delta_ticks = round(value * ENCODER_RESOLUTION / (360.0 * direction))
        raw_unwrapped = int(item["zero_position_raw"]) + delta_ticks
        raw = raw_unwrapped % ENCODER_RESOLUTION if item["continuous"] else raw_unwrapped

    if joint != "gripper" and not item["continuous"]:
        low, high = int(item["soft_min_raw"]), int(item["soft_max_raw"])
        if raw < low or raw > high:
            raise ValueError(f"{joint} 目标 raw={raw} 超出实测软限位 [{low}, {high}]")
    elif joint != "gripper" and (raw < 0 or raw > ENCODER_MAX):
        raise ValueError(f"{joint} 目标编码器值无效：{raw}")
    return int(raw)

def slider_bounds(joint):
    item = params["joints"][joint]
    if joint == "gripper":
        return 0.0, 100.0, "%"
    design_low, design_high = design_bounds(joint)
    if item["continuous"]:
        return design_low, design_high - 0.1, "°"
    endpoint_values = [
        raw_to_display_value(joint, item["soft_min_raw"]),
        raw_to_display_value(joint, item["soft_max_raw"]),
    ]
    calibrated_low, calibrated_high = min(endpoint_values), max(endpoint_values)
    lower = max(design_low, calibrated_low)
    upper = min(design_high, calibrated_high)
    if lower >= upper:
        raise ValueError(f"{joint} 最终设计范围与实测软限位没有有效交集。")
    return lower, upper, "°"

print("最终可用界面范围（设计范围 ∩ 实测软限位）：")
for joint in JOINT_ORDER:
    lower, upper, unit = slider_bounds(joint)
    print(f"{joint:8s}: {lower:8.2f} ～ {upper:8.2f} {unit}")

## 4. 连接舵机总线

连接后立即关闭所有舵机扭矩，并检查它们处于位置模式。该单元不会让机械臂运动。

In [ ]:
from lerobot.motors import Motor, MotorNormMode
from lerobot.motors.feetech import FeetechMotorsBus, OperatingMode

motors = {}
for joint in JOINT_ORDER:
    item = params["joints"][joint]
    norm_mode = MotorNormMode.RANGE_0_100 if joint == "gripper" else MotorNormMode.DEGREES
    motors[joint] = Motor(int(item["id"]), "sts3215", norm_mode)

bus = FeetechMotorsBus(port=PORT, motors=motors)
bus.connect(handshake=True)
bus.disable_torque(num_retry=2)

for joint in JOINT_ORDER:
    mode = int(bus.read("Operating_Mode", joint, normalize=False, num_retry=2))
    if mode != OperatingMode.POSITION.value:
        bus.disconnect(disable_torque=True)
        raise RuntimeError(f"{joint} 当前 Operating_Mode={mode}，不是位置模式0。请先检查舵机配置。")

positions = bus.sync_read("Present_Position", normalize=False, num_retry=2)
print("连接成功，全部舵机扭矩已关闭：")
for joint in JOINT_ORDER:
    print(f"{joint:8s} raw={int(positions[joint]):4d} display={raw_to_display_value(joint, positions[joint]):8.2f}")


## 5. 创建图形化单舵机控制面板

按钮说明：

- **读取状态**：读取当前位置、电压、温度和电流，不改变扭矩；
- **使能/保持**：先把当前位置写为目标，再使能该关节，避免突然跳到旧目标；
- **移动到目标**：使用当前滑块值发送位置命令；
- **目标设为零**：只把滑块设为 0，不立即运动；夹爪对应 0%；
- **卸力**：关闭该关节扭矩；
- **全部紧急卸力**：关闭所有舵机扭矩。

In [ ]:
if not bus.is_connected:
    raise RuntimeError("总线未连接，请先运行第 4 节。")

joint_controls = {}
enabled_joints = set()
global_status = widgets.HTML(value="<b>状态：</b>全部关节当前为卸力状态。")

def set_joint_enabled_style(joint, enabled):
    controls = joint_controls[joint]
    controls["enable_button"].button_style = "success" if enabled else ""
    controls["release_button"].button_style = "" if enabled else "warning"

def update_joint_status(joint, message, color="#333333"):
    joint_controls[joint]["status"].value = f"<span style='color:{color}'>{message}</span>"

def read_joint_feedback(joint, update_slider=False):
    raw = int(bus.read("Present_Position", joint, normalize=False, num_retry=2))
    value = raw_to_display_value(joint, raw)
    voltage = bus.read("Present_Voltage", joint, normalize=False, num_retry=2)
    temperature = bus.read("Present_Temperature", joint, normalize=False, num_retry=2)
    current = bus.read("Present_Current", joint, normalize=False, num_retry=2)
    if update_slider:
        slider = joint_controls[joint]["slider"]
        slider.value = max(slider.min, min(slider.max, value))
    unit = joint_controls[joint]["unit"]
    return raw, value, voltage, temperature, current, unit

def on_read_clicked(joint):
    try:
        raw, value, voltage, temperature, current, unit = read_joint_feedback(joint, update_slider=False)
        update_joint_status(
            joint,
            f"位置={value:.2f}{unit} / raw={raw}；电压raw={voltage}；温度={temperature}°C；电流raw={current}",
            "#0B6E4F",
        )
    except Exception as exc:
        update_joint_status(joint, f"读取失败：{type(exc).__name__}: {exc}", "#B00020")

def on_enable_clicked(joint):
    try:
        current_raw = int(bus.read("Present_Position", joint, normalize=False, num_retry=2))
        bus.write("Goal_Position", joint, current_raw, normalize=False, num_retry=2)
        bus.write("Goal_Velocity", joint, int(speed_widget.value), normalize=False, num_retry=2)
        bus.write("Acceleration", joint, int(acceleration_widget.value), normalize=False, num_retry=2)
        bus.enable_torque(joint, num_retry=2)
        enabled_joints.add(joint)
        set_joint_enabled_style(joint, True)
        update_joint_status(joint, f"已使能并保持当前位置 raw={current_raw}", "#0B6E4F")
    except Exception as exc:
        try:
            bus.disable_torque(joint, num_retry=2)
        except Exception:
            pass
        enabled_joints.discard(joint)
        set_joint_enabled_style(joint, False)
        update_joint_status(joint, f"使能失败：{type(exc).__name__}: {exc}", "#B00020")

def on_move_clicked(joint):
    if joint not in enabled_joints:
        update_joint_status(joint, "请先点击“使能/保持”，确认稳定后再移动。", "#B00020")
        return
    try:
        target_value = float(joint_controls[joint]["slider"].value)
        target_raw = display_value_to_raw(joint, target_value)
        bus.write("Goal_Velocity", joint, int(speed_widget.value), normalize=False, num_retry=2)
        bus.write("Acceleration", joint, int(acceleration_widget.value), normalize=False, num_retry=2)
        bus.write("Goal_Position", joint, target_raw, normalize=False, num_retry=2)
        unit = joint_controls[joint]["unit"]
        update_joint_status(joint, f"已发送目标 {target_value:.2f}{unit} / raw={target_raw}", "#005BBB")
    except Exception as exc:
        try:
            bus.disable_torque(joint, num_retry=2)
        except Exception:
            pass
        enabled_joints.discard(joint)
        set_joint_enabled_style(joint, False)
        update_joint_status(joint, f"移动失败并已卸力：{type(exc).__name__}: {exc}", "#B00020")

def on_release_clicked(joint):
    try:
        bus.disable_torque(joint, num_retry=2)
        enabled_joints.discard(joint)
        set_joint_enabled_style(joint, False)
        update_joint_status(joint, "已卸力。承重关节请保持支撑。", "#8A4B08")
    except Exception as exc:
        update_joint_status(joint, f"卸力命令失败：{type(exc).__name__}: {exc}", "#B00020")

def on_zero_clicked(joint):
    joint_controls[joint]["slider"].value = 0.0
    update_joint_status(joint, "目标滑块已设为 0；尚未发送运动命令。", "#555555")

def make_handler(function, joint):
    return lambda _button: function(joint)

speed_widget = widgets.IntSlider(
    value=DEFAULT_SPEED, min=50, max=1000, step=10, description="速度", continuous_update=False,
    style={"description_width": "60px"}, layout=widgets.Layout(width="360px")
)
acceleration_widget = widgets.IntSlider(
    value=DEFAULT_ACCELERATION, min=1, max=100, step=1, description="加速度", continuous_update=False,
    style={"description_width": "60px"}, layout=widgets.Layout(width="360px")
)

rows = []
for joint in JOINT_ORDER:
    lower, upper, unit = slider_bounds(joint)
    current_raw = int(bus.read("Present_Position", joint, normalize=False, num_retry=2))
    initial_value = max(lower, min(upper, raw_to_display_value(joint, current_raw)))
    slider = widgets.FloatSlider(
        value=initial_value, min=lower, max=upper, step=0.5 if joint == "gripper" else 0.1,
        description=f"{joint} 目标", readout_format=".1f", continuous_update=False,
        style={"description_width": "75px"}, layout=widgets.Layout(width="520px"),
    )
    read_button = widgets.Button(description="读取状态", icon="refresh")
    enable_button = widgets.Button(description="使能/保持", icon="lock")
    move_button = widgets.Button(description="移动到目标", icon="play", button_style="primary")
    zero_button = widgets.Button(description="目标设为零", icon="crosshairs")
    release_button = widgets.Button(description="卸力", icon="unlock", button_style="warning")
    status = widgets.HTML(value=f"<span style='color:#555555'>当前位置 raw={current_raw}；当前为卸力状态。</span>")

    joint_controls[joint] = {
        "slider": slider, "unit": unit, "status": status,
        "read_button": read_button, "enable_button": enable_button,
        "move_button": move_button, "zero_button": zero_button, "release_button": release_button,
    }
    read_button.on_click(make_handler(on_read_clicked, joint))
    enable_button.on_click(make_handler(on_enable_clicked, joint))
    move_button.on_click(make_handler(on_move_clicked, joint))
    zero_button.on_click(make_handler(on_zero_clicked, joint))
    release_button.on_click(make_handler(on_release_clicked, joint))

    title = widgets.HTML(value=f"<h4 style='margin:4px 0'>{joint} — {params['joints'][joint]['function']}（{unit}）</h4>")
    buttons = widgets.HBox([read_button, enable_button, move_button, zero_button, release_button])
    box = widgets.VBox([title, slider, buttons, status], layout=widgets.Layout(
        border="1px solid #C9D2DC", padding="8px", margin="5px 0", width="100%"
    ))
    rows.append(box)

def release_all(_button=None):
    try:
        bus.disable_torque(num_retry=2)
        enabled_joints.clear()
        for joint in JOINT_ORDER:
            set_joint_enabled_style(joint, False)
            update_joint_status(joint, "已由“全部紧急卸力”关闭扭矩。", "#B00020")
        global_status.value = "<b style='color:#B00020'>全部舵机已卸力。承重关节请保持支撑。</b>"
    except Exception as exc:
        global_status.value = f"<b style='color:#B00020'>卸力命令失败：{type(exc).__name__}: {exc}。请立即切断电源。</b>"

emergency_button = widgets.Button(
    description="全部紧急卸力", icon="stop", button_style="danger",
    layout=widgets.Layout(width="220px", height="42px"),
)
emergency_button.on_click(release_all)

header = widgets.VBox([
    widgets.HTML(value="<h3>STS3215 单舵机控制面板</h3><p>滑块不会自动发送命令；必须先使能，再点击移动。</p>"),
    widgets.HBox([speed_widget, acceleration_widget]),
    widgets.HBox([emergency_button, global_status]),
])
display(widgets.VBox([header] + rows, layout=widgets.Layout(width="100%")))


## 6. 推荐测试顺序

每个关节按以下顺序测试：

1. 用手或支架支撑相关连杆；
2. 点击“读取状态”，检查当前位置、温度和电压反馈；
3. 点击“使能/保持”，观察关节是否稳定且没有跳动；
4. 将目标只改变 2°～5°（夹爪改变 2%～5%）；
5. 点击“移动到目标”，确认方向和运动正常；
6. 逐步扩大测试范围，但不要一开始直接移动到限位；
7. 测试结束后点击该关节“卸力”，承重关节要继续托住；
8. 全部测试完成后运行最后的断开单元。

## 7. 关闭扭矩并断开串口

关闭 Notebook、重启 kernel 或拔下 USB 前，先运行下一格。

In [ ]:
if "bus" in globals() and bus.is_connected:
    bus.disable_torque(num_retry=2)
    bus.disconnect(disable_torque=True)
    if "enabled_joints" in globals():
        enabled_joints.clear()
    print("全部舵机已卸力，串口已断开。现在可以关闭舵机电源。")
else:
    print("总线当前未连接。")


## 8. 常见问题

**控件不显示**：确认使用 Jupyter/VS Code Notebook 并选择 `lerobot` kernel；重新运行环境单元。

**点击按钮没有运动**：先点击对应关节的“使能/保持”；检查状态栏是否有错误。

**提示串口被占用**：先运行第7节断开；关闭其他 Notebook、串口助手或旧 kernel。

**使能时发生明显跳动**：立即紧急卸力。检查 ID、零位、舵盘安装和是否有其他程序同时发送命令。

**运动到错误方向**：立即卸力。当前参数文件中的 `direction` 必须与固定安装方向一致。

**J1/J4 接近 ±180°或 J6 接近 0°/360°时反向绕行**：这是单圈编码器边界。先回到远离边界的位置再测试；不要让外部线缆缠绕。

**温度或电流持续升高**：卸力并断电，检查机械干涉、负载、供电和舵机保护参数。

**界面范围比实测范围小**：这是正常的。控制界面使用最终设计范围与实测软限位的交集，不能因为机构实测还能继续转动就超过最终运动学设计范围。